# Habits: per-move features from engine analysis

**Milestone:** "Estimate player habits from game history" (README Teaching). Gambit estimates a
player's level from **habits** (recurring mistakes, missed tactics, risky or passive tendencies)
rather than from results.

For every move the player made we ask the engine (a full-strength *analyst*):

* **Centipawn loss (CPL)**: eval of the best move minus eval of the played move, both from the
  player's point of view, computed at the same depth (`root_moves=[played]`), capped at `cpl_cap`.
* **Classification**: inaccuracy / mistake / blunder by CPL thresholds in game.yaml
  (`teaching.cpl_thresholds`, lichess-like defaults, to be tuned).
* **Missed forcing move**: the best move was a capture or check and the player did not find it
  (and lost at least an inaccuracy's worth). Specific tactic detectors (fork, pin, ...) are TODO.
* **Phase**: opening / middlegame / endgame (config thresholds).
* **Time use**: think time from the PGN `[%emt]` annotation (our logger writes it) or the robot log.

Per player these aggregate into a `HabitProfile`.

**TODO checklist**
- [x] CPL, classification, phase, per-move analysis, profile aggregation.
- [ ] Tactic detectors (`TACTIC_DETECTORS`): fork, pin/skewer, hanging piece, back-rank.
- [ ] Risky vs passive tendencies (README): e.g. share of moves that reduce king safety, trade
      offers declined, pawn-storm frequency. Define, then validate they are stable per player.
- [ ] Use win-probability loss instead of raw CPL in lopsided positions (CPL of 300 at +900
      barely matters); lichess uses a logistic transform.

In [ ]:
%run ../../gambit_game/notebooks/engine.ipynb

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass, field
from typing import Callable, Iterable, Optional

import chess
import chess.pgn
import pandas as pd

TEACH_CFG = GAME_CFG["teaching"]
CPL_THRESHOLDS = TEACH_CFG["cpl_thresholds"]
CLASSES = ("best", "good", "inaccuracy", "mistake", "blunder")

## CPL and classification

In [ ]:
def centipawn_loss(best_cp: int, played_cp: int, cap: int = TEACH_CFG["cpl_cap"]) -> int:
    """Both scores from the mover's POV. Never negative (the engine's best can be underestimated)."""
    return int(min(cap, max(0, best_cp - played_cp)))


def classify_cpl(cpl: int, is_best: bool = False, thresholds: dict = CPL_THRESHOLDS) -> str:
    if is_best or cpl == 0:
        return "best"
    if cpl >= thresholds["blunder"]:
        return "blunder"
    if cpl >= thresholds["mistake"]:
        return "mistake"
    if cpl >= thresholds["inaccuracy"]:
        return "inaccuracy"
    return "good"

## Game phase

Simple and explainable: opening by move number, endgame by remaining non-pawn material
(N = B = 3, R = 5, Q = 9, both sides together), middlegame otherwise.

In [ ]:
PHASE_VALUE = {chess.KNIGHT: 3, chess.BISHOP: 3, chess.ROOK: 5, chess.QUEEN: 9}


def non_pawn_material(board: chess.Board) -> int:
    return sum(PHASE_VALUE.get(p.piece_type, 0) for p in board.piece_map().values())


def game_phase(board: chess.Board, cfg: dict = TEACH_CFG["phase"]) -> str:
    if non_pawn_material(board) <= cfg["endgame_max_material"]:
        return "endgame"
    if board.fullmove_number <= cfg["opening_max_fullmove"]:
        return "opening"
    return "middlegame"

## Tactic detectors (TODO)

Each detector gets the position before the move, the engine's best line and the move played, and
returns `True` if the player missed that tactic. They plug into `analyse_move(detectors=...)`.

In [ ]:
def detect_missed_fork(board: chess.Board, best: Candidate, played: chess.Move) -> bool:
    """TODO: best move (or the PV's 2nd own move) attacks two+ valuable enemy pieces at once
    (board.attacks after pushing the move), and the played move did not."""
    raise NotImplementedError("detect_missed_fork")


def detect_missed_pin(board: chess.Board, best: Candidate, played: chess.Move) -> bool:
    """TODO: best move creates a pin/skewer (board.is_pinned on the opponent after the move)."""
    raise NotImplementedError("detect_missed_pin")


def detect_hanging_piece(board: chess.Board, best: Candidate, played: chess.Move) -> bool:
    """TODO: after the played move, one of the player's pieces is attacked and not defended
    (or defended by less than its value) and the engine's reply captures it."""
    raise NotImplementedError("detect_hanging_piece")


TACTIC_DETECTORS: dict[str, Callable] = {
    "fork": detect_missed_fork,
    "pin": detect_missed_pin,
    "hanging_piece": detect_hanging_piece,
}

## Per-move analysis

In [ ]:
@dataclass
class MoveAnalysis:
    game_id: str
    ply: int
    color: str                        # "white" | "black" (the player who moved)
    fen: str                          # position before the move
    uci: str
    best_uci: str
    best_cp: int                      # mover POV
    played_cp: int                    # mover POV
    cpl: int
    classification: str
    phase: str
    best_was_forcing: bool            # best move is a capture or check
    missed_forcing: bool
    think_time_s: Optional[float] = None
    tactics_missed: list = field(default_factory=list)


def analyse_move(board: chess.Board, move: chess.Move, analyst: EngineLike, game_id: str = "",
                 think_time_s: Optional[float] = None, multipv: int = 1,
                 detectors: Iterable[str] = ()) -> MoveAnalysis:
    cands = analyst.analyse(board, multipv=multipv)
    best = cands[0]
    played = next((c for c in cands if c.move == move), None)
    if played is None:
        played = analyst.analyse(board, multipv=1, root_moves=[move])[0]
    is_best = move == best.move or played.score_cp >= best.score_cp
    cpl = 0 if is_best else centipawn_loss(best.score_cp, played.score_cp)
    cls = classify_cpl(cpl, is_best)
    forcing = board.is_capture(best.move) or board.gives_check(best.move)
    missed = forcing and not is_best and cpl >= CPL_THRESHOLDS["inaccuracy"]
    tactics = [name for name in detectors if TACTIC_DETECTORS[name](board, best, move)]
    return MoveAnalysis(game_id, board.ply(), "white" if board.turn else "black", board.fen(),
                        move.uci(), best.move.uci(), best.score_cp, played.score_cp, cpl, cls,
                        game_phase(board), forcing, missed, think_time_s, tactics)


def player_color_in_game(game: chess.pgn.Game, player: str) -> Optional[chess.Color]:
    """Which colour `player` (case-insensitive name) played in a PGN game, or None."""
    p = player.lower()
    if game.headers.get("White", "").lower() == p:
        return chess.WHITE
    if game.headers.get("Black", "").lower() == p:
        return chess.BLACK
    return None


def analyse_game(game: chess.pgn.Game, analyst: EngineLike, color: chess.Color,
                 game_id: Optional[str] = None, multipv: int = 1,
                 detectors: Iterable[str] = ()) -> list[MoveAnalysis]:
    """Analyse every move `color` made in a PGN game. Think time from `[%emt]` if present."""
    gid = game_id or game.headers.get("Site", "") or game.headers.get("Event", "game")
    board = game.board()
    out = []
    for node in game.mainline():
        move = node.move
        if board.turn == color:
            out.append(analyse_move(board, move, analyst, gid, node.emt(), multipv, detectors))
        board.push(move)
    return out

## HabitProfile: aggregate over many games

In [ ]:
@dataclass
class HabitProfile:
    player: str
    n_games: int
    n_moves: int
    acpl: float
    per_game_acpl: list
    class_rates: dict                 # classification -> share of moves
    acpl_by_phase: dict               # phase -> ACPL
    moves_by_phase: dict
    missed_forcing_rate: float        # missed forcing moves / positions where best was forcing
    n_forcing_positions: int
    mean_think_time_s: Optional[float]

    def to_dict(self) -> dict:
        return asdict(self)


def analyses_to_frame(analyses: list[MoveAnalysis]) -> pd.DataFrame:
    return pd.DataFrame([asdict(a) for a in analyses])


def build_habit_profile(player: str, analyses: list[MoveAnalysis]) -> HabitProfile:
    df = analyses_to_frame(analyses)
    if df.empty:
        return HabitProfile(player, 0, 0, float("nan"), [], {}, {}, {}, float("nan"), 0, None)
    forcing = df[df.best_was_forcing]
    tt = df.think_time_s.dropna() if "think_time_s" in df else pd.Series(dtype=float)
    return HabitProfile(
        player=player,
        n_games=int(df.game_id.nunique()),
        n_moves=len(df),
        acpl=float(df.cpl.mean()),
        per_game_acpl=[float(v) for v in df.groupby("game_id", sort=False).cpl.mean()],
        class_rates={c: float((df.classification == c).mean()) for c in CLASSES},
        acpl_by_phase={k: float(v) for k, v in df.groupby("phase").cpl.mean().items()},
        moves_by_phase={k: int(v) for k, v in df.groupby("phase").size().items()},
        missed_forcing_rate=float(forcing.missed_forcing.mean()) if len(forcing) else float("nan"),
        n_forcing_positions=len(forcing),
        mean_think_time_s=float(tt.mean()) if len(tt) else None,
    )

## Self-check (MaterialEngine as a stand-in analyst; real analysis uses Stockfish)

In [ ]:
import io

assert centipawn_loss(50, -30) == 80 and centipawn_loss(10, 40) == 0 and centipawn_loss(9000, -9000) == TEACH_CFG["cpl_cap"]
assert [classify_cpl(c) for c in (0, 20, 60, 150, 400)] == ["best", "good", "inaccuracy", "mistake", "blunder"]
assert game_phase(chess.Board()) == "opening"
assert game_phase(chess.Board("4k3/8/8/8/8/8/4P3/R3K3 w - - 0 40")) == "endgame"
assert game_phase(chess.Board("r1bqkbnr/pppppppp/2n5/8/8/2N5/PPPPPPPP/R1BQKBNR w KQkq - 0 15")) == "middlegame"

# Scholar's mate: black's 3...Nf6?? allows Qxf7# -> blunder; white's Qxf7# is best.
pgn = io.StringIO('[White "alice"]\n[Black "bob"]\n\n1. e4 {[%emt 0:00:04]} e5 2. Qh5 Nc6 3. Bc4 Nf6 4. Qxf7# 1-0\n')
game = chess.pgn.read_game(pgn)
analyst = MaterialEngine(depth=2)
black = analyse_game(game, analyst, player_color_in_game(game, "BOB"), "g1")
white = analyse_game(game, analyst, player_color_in_game(game, "alice"), "g1")
assert [a.uci for a in black] == ["e7e5", "b8c6", "g8f6"]
assert black[-1].classification == "blunder" and black[-1].cpl == TEACH_CFG["cpl_cap"], black[-1]
assert white[-1].classification == "best" and white[-1].best_was_forcing
assert white[0].think_time_s == 4.0
prof = build_habit_profile("bob", black)
assert prof.n_moves == 3 and prof.n_games == 1 and prof.class_rates["blunder"] == 1 / 3
assert abs(prof.acpl - sum(a.cpl for a in black) / 3) < 1e-9
print("habits self-check passed")